**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**05 | Frozen Test and Research Synthesis**

![Frozen Test and Research Synthesis](../../images/05_frozen_test_and_research_synthesis.svg)

**Objective:** Apply the Development-selected IMU event detector to unseen participants once, quantify event timing and coverage, and interpret participant-level uncertainty.

**Dataset:** Camargo et al. (2021), source-validated original-time paired IMU and independent force-derived contact references.

**Notebook scope:** Development/Test boundaries and stated measurement units are preserved. Numerical results are only established after original-data execution.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Load the saved Development-selected configuration and the participant-disjoint Test records. No tuning or source-dependent fitting occurs in this Notebook.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from gait_experiments import (load_trial_collection,
    evaluate_trial_collection, FIXED_BASELINE)
from gait_evaluation import (participant_metrics, paired_subject_bootstrap)

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
output_dir = study_dir / "output"
cache_dir = root / "data/processed/camargo/paired_trials"
selection = pd.read_csv(output_dir / "04_selected_configuration.csv")
if len(selection) != 1:
    raise ValueError("Notebook 04 must save exactly one selected configuration.")
choice = selection.iloc[0]
model_name = str(choice.selected_model_id)
config = json.loads(choice.configuration_json)
tolerance = float(choice.match_tolerance_s)
print("Development-selected recipe:", model_name, config)
print("Matching tolerance:", tolerance, "seconds")


**02. Verify Frozen Participant Roles and Source Inputs**

Test includes only participants reserved in Notebook 02, and their original-time numerical inputs must already exist. Neither model choice nor detection threshold is adjusted after inspecting Test scores.


In [ ]:
# ========================= 02. Verify Reserved Test ==========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
eligible = pd.read_csv(output_dir / "02_eligible_trials.csv")
roles = pd.read_csv(output_dir / "02_participant_roles.csv")
test_people = set(roles.loc[roles.role=="Test", "subject_id"])
development_people = set(roles.loc[roles.role=="Development","subject_id"])
if not test_people or (test_people & development_people):
    raise ValueError("Missing or overlapping participant roles.")
test = eligible.loc[eligible.subject_id.isin(test_people)].copy()
if set(test.subject_id) != test_people:
    raise ValueError("Some reserved Test people have no eligible source trials.")
trials = load_trial_collection(cache_dir, test)
print("Held-out participants:", sorted(test_people),
      "eligible Test trials:", len(trials))


**03. Evaluate Frozen Model and Fixed Reference Baseline**

Both detectors are scored on the identical reserved Test trials. The primary measurement is the selected model's participant-mean matched IC timing MAE with explicit sensitivity, precision, misses and extras. B0 remains a predefined descriptive comparator.


In [ ]:
# ======================== 03. Evaluate Frozen Models =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
results = []
# Compare two distinct recipes only when Development selected B1.
methods = [("B0_fixed", FIXED_BASELINE)]
if model_name != "B0_fixed":
    methods.append(("selected_" + model_name, config))
for model_id, method in methods:
    trial_scores = evaluate_trial_collection(
        trials, method, model_id, tolerance
    )
    results.append(trial_scores)
test_scores = pd.concat(results, ignore_index=True)

# ------------------------- 2. Save Scientific Tables -------------------------
test_scores.to_csv(output_dir / "05_test_trial_metrics.csv", index=False)
display(test_scores.head())


**04. Calculate Person-Balanced Test Evidence**

Summarise at the independent participant level, not the number of events. Timing MAE must always be interpreted with the event-detection counts; no matched events produce undefined timing error rather than a spurious zero.


In [ ]:
# ====================== 04. Summarise Test Participants ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
p = []
for model_id, frame in test_scores.groupby("model_id"):
    p.append(participant_metrics(frame).assign(model_id=model_id))
person_results = pd.concat(p, ignore_index=True)
person_results.to_csv(
    output_dir / "05_test_participant_metrics.csv", index=False
)
summary = (
    person_results.groupby("model_id", as_index=False)
        .agg(mean_subject_timing_mae_ms=("timing_mae_ms", "mean"),
             mean_subject_sensitivity=("sensitivity", "mean"),
             mean_subject_precision=("precision", "mean"),
             n_participants=("subject_id", "nunique"))
)

# ------------------------- 2. Save Scientific Tables -------------------------
summary.to_csv(output_dir / "05_test_model_summary.csv", index=False)
display(summary)


**05. Paired Uncertainty and Event Error Diagnostics**

Describe paired whole-person uncertainty. A small number of Test participants limits precision even when many correlated strides or events are recorded. No inferential superiority is asserted from descriptive intervals.


In [ ]:
# ======================== 05. Paired Test Uncertainty ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
if model_name == "B0_fixed":
    # No paired model effect exists when the fixed baseline was selected.
    uncertainty = pd.DataFrame([{
        "comparator": "B0_fixed", "selected_model": model_name,
        "n_participants": len(test_people), "mean_difference_ms": np.nan,
        "lower_ms": np.nan, "upper_ms": np.nan,
        "comparison_note": "Same Development-selected model; no paired contrast"
    }])
else:
    delta = paired_subject_bootstrap(
        person_results, "B0_fixed", "selected_" + model_name,
        draws=2000, seed=42
    )
    uncertainty = pd.DataFrame([{
        "comparator": "B0_fixed", "selected_model": model_name,
        "comparison_note": "Selected B1 minus fixed B0 timing MAE", **delta
    }])

# ------------------------- 2. Save Scientific Tables -------------------------
uncertainty.to_csv(
    output_dir / "05_paired_participant_uncertainty.csv", index=False
)
display(uncertainty)


**06. Visualise Matched Test Performance and Coverage**

Display participant-level observed errors and sensitivity. The plotted points are actual recorded Test outputs, not simulated gait signals.


In [ ]:
# ======================== 06. Visualise Test Evidence ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
fig, axes = plt.subplots(1, 2, figsize=(11.8,4.4))
for model, frame in person_results.groupby("model_id"):
    axes[0].scatter([model] * len(frame), frame.timing_mae_ms,
                    alpha=0.65, label=model)
    axes[1].scatter([model] * len(frame), frame.sensitivity,
                    alpha=0.65, label=model)
axes[0].set(title="Held-out participant event timing",
            ylabel="Matched IC timing MAE (ms)")
axes[1].set(title="Held-out event coverage",
            ylabel="IC sensitivity")
for ax in axes:
    ax.grid(alpha=0.2)

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / "05_test_participant_comparison.png")
plt.show()


**07. Final Scientific Interpretation and Boundaries**

The participant-independent Test supports only an offline within-dataset gait-event comparison for the available Camargo conditions. It cannot establish patient performance, real-time deployment or cross-device generalisation. The reported final Test result must come from the tables written above, not from planned values.


In [ ]:
# ======================== 07. Interpret Final Results ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
print("Selected Development method:", model_name)
print("Independent Test participants:", len(test_people))
print("Observed eligible Test trials:", len(trials))
print("Primary recorded Test file:",
      output_dir / "05_test_participant_metrics.csv")
print("Interpret timing MAE beside sensitivity, precision and missed events.")
print("No raw-force signal was used to fit a Test-time IMU detector.")
